# Simple EDA — German grid load and residual load

**Spec:** [`specs/01-Simple-EDA.md`](../specs/01-Simple-EDA.md) ·
**Data:** `data/smard.csv` (SMARD / Bundesnetzagentur, hourly, region DE)

## What this notebook is for

Understand the SMARD dataset well enough to make informed modeling decisions for the
1-day-ahead **residual load** forecast, and describe — descriptively, without defining
thresholds — where the extreme residual load cases that motivate the project actually sit.

It answers five questions:

1. Is the dataset complete, correctly typed, and continuous enough to be treated as an hourly
   time series?
2. Are our aggregation helpers correct, and what exactly do our calendar conventions mean?
3. What are the trend, seasonal and calendar structures in each series?
4. How do the series relate to each other, and which relations are candidate features?
5. What does the `residual_load` distribution look like, and how do its tails behave in time?

## How to read it

- **§6 is a contract, not a private choice.** The week convention, the season definition, the
  reporting units and the descriptive-slice policy fixed there are inherited by
  [`specs/02-Deep-EDA.md`](../specs/02-Deep-EDA.md) rather than re-derived.
- **Every plot** carries a title, axis labels and explicit units — MWh, average MW or MWh/day,
  never an unlabelled number — and is followed by one to three sentences saying what it shows.
- **No thresholds.** This notebook does not define a risk flag, a cut-off or a labelled column.
  Where it shows "the tail hours" it selects them by rank, for description only (§6.5).
- **Units.** Values are energy per hourly interval in MWh. Over an hourly interval that number
  is also the average power in MW, which is why the hourly mean of a series and its average MW
  are the same number — §6.4 makes that explicit rather than leaving it to be inferred.

---

## 1 · Setup

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](API-connection.ipynb) top to bottom — it pulls the
SMARD API (no key required) and writes the file in German Excel CSV format
(`sep=";"`, `decimal=","`, `utf-8-sig`).

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0  # this notebook draws ~30 figures on purpose

# Works whether the kernel starts in notebooks/ (Jupyter) or at the repo root (nbconvert).
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")

---

## 2 · Helpers

`period_mean`, `style_timeseries` and `seasonal_plot` are **copied** from
[`notebooks/EDA-robert.ipynb`](EDA-robert.ipynb), which this spec does not modify. Three
changes apply to the copies here:

1. `ylabel` becomes a **required** argument in `style_timeseries` and `seasonal_plot`. The
   originals default it to `"(MWh)"`, which silently violates the reporting convention of §6.5.
2. `seasonal_plot` **honours** `ylabel`. The original accepts it, documents it, and then
   hard-codes `ax.set_ylabel("MWh")`.
3. `period_mean`'s edge rule moves into `_complete_periods` so it exists in exactly one place,
   shared with the new `period_energy`. The arithmetic is unchanged — §6.3 proves it.

`period_mean`'s docstring also drops the phrase *"the first and last one"*. The rule is **drop
periods the data does not fully cover**, which is not the same thing: this record starts exactly
on a month boundary, so January 2022 is complete and only the trailing month is dropped. The
code was always right; the prose was not.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(time_series, freq):
    """Mean of `time_series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last":
    see the correctness test in section 6.3.

    Copied from notebooks/EDA-robert.ipynb; the edge rule was factored into `_complete_periods`
    without changing the result.
    """
    agg = time_series.groupby(time_series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(time_series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(time_series, freq, drop_incomplete=True):
    """Per-period aggregate of `time_series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW). Deliberately not
        ``mwh_per_day / 24``: a month containing the spring DST switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so section 6.4's comparison table needs no second copy of
        this arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`. Pass
    ``drop_incomplete=False`` to keep them — only the 6.4 table does, because it has to *show*
    the period the rule discards.
    """
    grouped = time_series.groupby(time_series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(time_series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required (not defaulted to "(MWh)" as in the original): every plot must state
    whether it shows MWh, average MW or MWh/day. See the reporting convention in section 6.5.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over 41k rows — minutes of runtime, meaningless band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied — the
            original hard-coded "MWh" and ignored this argument.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

---

## 3 · Load and prepare

The CSV is German Excel format, so every numeric column arrives as text with a comma decimal
separator. The failure mode to guard against is silent: unconverted columns land as a string
dtype, every aggregate still computes something, and every number is wrong. The dtype assertion
below is the guard.

The flat, `RangeIndex`ed frame is called `raw` and is **deleted** at the end of the loading
cells. Everything downstream uses `ts`, so the two cannot drift apart.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them. Note the inconsistent
# capitalisation in the source ("Grid Load" vs "Forecast Grid load") — reproduced deliberately.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid load": "fc_grid_load",
    "Forecast Residual Load": "fc_res",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

ts = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cells

ts.head(3)

In [ ]:
print(f"shape           : {ts.shape[0]:,} rows x {ts.shape[1]} columns")
print(f"index           : {ts.index.min()}  ->  {ts.index.max()}")
print(f"index monotonic : {ts.index.is_monotonic_increasing}, unique: {ts.index.is_unique}")
display(ts.dtypes.to_frame("dtype"))

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(pd.api.types.is_float_dtype(ts[c]) for c in COLUMNS.values()), ts.dtypes

# Snapshot for the self-check in section 11: re-asserted at the end, so a cell inserted anywhere
# in between that mutates `ts` is caught regardless of which vintage of the CSV was loaded.
LOADED = {"rows": len(ts), "start": ts.index.min(), "end": ts.index.max()}

display(ts.describe().T)

41 107 hourly rows spanning 2022-01-01 00:00 to 2026-09-09 23:00, all eight series `float64`,
nothing obviously degenerate in `describe()`. The one number worth pausing on is
`residual_load`'s minimum: it is **negative**, which is valid data (renewable oversupply), not
an error. That rules out log scales and log transforms for this series throughout.

---

## 4 · Derived columns and the `SERIES` constant

All derived columns are defined **here, in one place**, immediately after loading. Two of them
are needed by the data quality audit itself (`renewables` for the identity check, `hour` for the
night-solar check), so they cannot wait for the section that first plots them.

`SERIES` names the eight data columns. Without it, `ts.corr()` in §9 would silently pull the
derived columns into the correlation heatmap, and `describe()` would report on `year` and `dow`
as though they were measurements.

> **On "nine series".** The spec says nine; the CSV has eight numeric columns. Its Data table
> has nine rows only because it counts `timestamp`. Eight it is — `renewables` is derived, and
> appears in §9.2 where its relation to `residual_load` is the actual subject rather than a
> tautology cluttering a heatmap.

`season`/`season_year` and `spans_gap` are created here with everything else; the *reasoning*
behind them lives where it is used — the gap narrative in §5.2, the season contract in §6.5.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_res",
]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter. Rationale in 6.5.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

ts["renewables"] = ts[["wind_on", "wind_off", "solar"]].sum(axis=1)
ts["year"] = ts.index.year
ts["month"] = ts.index.month
ts["hour"] = ts.index.hour
ts["dow"] = ts.index.dayofweek
ts["is_weekend"] = ts.index.dayofweek >= 5
ts["date"] = ts.index.date
ts["season"] = pd.Categorical(
    ts.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
ts["season_year"] = ts.index.year + (ts.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
ts["spans_gap"] = ts.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(ts.columns) == SERIES + DERIVED, list(ts.columns)
print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {ts.shape[1]} columns")
ts[DERIVED].head(3)

`ts` now carries exactly the eight data columns plus ten declared derived ones, and the assertion
above is what keeps that true. The closing self-check in §11 re-runs it against the same two
lists — which is the mechanical proof that no flag or label column crept in along the way.

One consequence to keep in mind for the rest of the notebook: `date` is object dtype, so a bare
`ts.groupby(...).mean()` now raises under pandas 3. Every aggregation from here on names its
columns explicitly.

---

## 5 · Data quality audit

Six checks: coverage, gaps, duplicates, missing values, value ranges, and the residual load
identity. The governing rule for this whole section is that **nothing is repaired**. Gaps are
found, named and documented; they are not filled, interpolated or reindexed away. How to handle
them is a modeling decision, and it belongs to the modeling spec, not to an audit.

### 5.1 · Coverage

In [ ]:
full_index = pd.date_range(ts.index.min(), ts.index.max(), freq="h")

print(f"first timestamp          : {ts.index.min()}")
print(f"last timestamp           : {ts.index.max()}")
print(f"rows in file             : {len(ts):,}")
print(f"complete hourly index    : {len(full_index):,}")
print(f"difference               : {len(full_index) - len(ts):,} hours missing")

The file is **5 hours short** of a complete hourly index over its own span: 41 107 rows where
41 112 would be needed. That is small enough to be invisible in any aggregate and large enough to
break anything that assumes a fixed row-to-hour mapping. §5.2 identifies every one of them.

### 5.2 · Gaps, and the two different ways this record loses hours

In [ ]:
missing_hours = full_index.difference(ts.index)
N_GAPS = len(missing_hours)

display(
    pd.DataFrame(
        {"weekday": missing_hours.day_name(), "hour_label": missing_hours.hour},
        index=missing_hours,
    ).rename_axis("missing timestamp")
)

# `spans_gap` marks the row FOLLOWING each gap; the two views must agree.
assert (ts.index[ts["spans_gap"]] == missing_hours + pd.Timedelta("1h")).all()
print(f"{N_GAPS} missing hours, and spans_gap flags exactly the {ts['spans_gap'].sum()} rows after them\n")

print("the 2025 spring switch, hour by hour -- 01:00 is followed directly by 03:00:")
display(ts.loc["2025-03-30 00:00":"2025-03-30 04:00", ["grid_load", "residual_load", "spans_gap"]])

All five missing timestamps are a **Sunday at 02:00 in late March** — the spring DST switch. At
02:00 CET the clock jumps to 03:00 CEST, so the wall-clock hour labelled 02:00 does not exist on
those days and the file correctly has no row for it. This is not missing data; it is a missing
*hour*.

The second, less obvious loss is the **autumn DST fold**, and it is a genuinely different effect
that no gap search can find:

In [ ]:
autumn_switches = pd.DatetimeIndex(
    [
        pd.date_range(f"{y}-10-01", f"{y}-10-31", freq="W-SUN")[-1]
        for y in range(ts.index.year.min(), ts.index.year.max() + 1)
    ]
)
autumn_switches = autumn_switches[autumn_switches <= ts.index.max()]

days = ts.index.normalize()
display(
    pd.DataFrame(
        {
            "rows_in_file": [(days == d).sum() for d in autumn_switches],
            "rows_labelled_02h": [((days == d) & (ts.index.hour == 2)).sum() for d in autumn_switches],
            "true_local_hours": 25,
        },
        index=autumn_switches.date,
    ).rename_axis("autumn switch date")
)

Each autumn switch day really has **25** local hours — 02:00 occurs twice, once at CEST and once
at CET — but the file carries **24 rows with a single 02:00 row**. SMARD has already collapsed
the repeated hour rather than emitting it twice, so one physical hour per autumn switch is merged
away.

The consequence matters for the rest of the notebook: the fold produces **neither an index gap
nor a duplicate timestamp**, so `missing_hours` and the duplicate check in §5.3 are both blind to
it. Four autumn switches sit in this record against five spring ones, because the data ends on
2026-09-09, before that year's October switch.

**Nothing above is repaired.** No gap is filled, interpolated or reindexed away anywhere in this
notebook: `ts` keeps exactly the 41 107 rows it was loaded with, and §11 asserts that
mechanically. What the gaps mean for `.diff()`, `.shift()`, rolling windows and the ACF is flagged
at each site where it bites.

### 5.3 · Duplicate timestamps

In [ ]:
print(f"duplicate timestamps: {ts.index.duplicated().sum()}")

Zero, as expected — and the reason is precisely the collapse described in §5.2. A dataset that
emitted the autumn fold honestly *would* show one duplicate per autumn switch. So this zero is
evidence that SMARD pre-processed the fold, **not** evidence that the record is hour-complete.
Read together with §5.1, the two results say: no hour appears twice, and five hours are absent.

### 5.4 · Missing values

In [ ]:
display(
    pd.DataFrame(
        {
            "n_missing": ts[SERIES].isna().sum(),
            "share_%": (ts[SERIES].isna().mean() * 100).round(3),
        }
    )
)

**Zero missing values in all eight series.** An explicit zero is a finding, not an absence of
one: it means no imputation strategy is needed for the columns themselves, and every NaN that
turns up later in the notebook is one *we* created — by a `.shift()`, a lag or an incomplete
aggregation period — rather than one that arrived with the data.

### 5.5 · Value ranges

A naive range check drowns in false positives, so the night-solar test carries an explicit
tolerance: night-time `solar` must be below **0.5 % of the series maximum**, not exactly zero.
The justification is quantified immediately after the check.

In [ ]:
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
solar_tol = 0.005 * ts["solar"].max()
night = ts["hour"].isin(NIGHT_HOURS)

checks = {
    "wind_off < 0": ts["wind_off"] < 0,
    "wind_on < 0": ts["wind_on"] < 0,
    "solar < 0": ts["solar"] < 0,
    "grid_load <= 0": ts["grid_load"] <= 0,
    f"night solar > {solar_tol:,.1f} MWh": night & (ts["solar"] > solar_tol),
}

print(f"solar max {ts['solar'].max():,.1f} MWh -> tolerance {solar_tol:,.1f} MWh")
print(f"night hours defined as {NIGHT_HOURS}\n")

for name, mask in checks.items():
    print(f"{name:<38} {mask.sum():>6} violations")
    if mask.any():
        display(ts.loc[mask, SERIES])  # list every real violation by timestamp

print(
    f"\nnight solar: max {ts.loc[night, 'solar'].max():,.2f} MWh, "
    f"{(ts.loc[night, 'solar'] == 0).sum():,} of {night.sum():,} night hours are exactly zero"
)

**No violations.** Generation is never negative, `grid_load` is always strictly positive, and no
night hour comes anywhere near the tolerance — the largest night-time solar value in the record
is about **147 MWh** against a **290 MWh** threshold, i.e. half of it.

The tolerance earns its place: only about 400 of the ~10 300 night hours are *exactly* zero, so a
strict `solar == 0` night test would report roughly **9 900 "violations"** and invite a
zero-inflation claim that the data does not support. Those few MWh are measurement and reporting
noise, not generation.

### 5.6 · The residual load identity

The project's target is `residual_load`. Before building anything on it, confirm empirically what
it *is* — and report the **distribution** of the difference rather than a yes/no, so that
two-decimal CSV rounding is not mistaken for a discrepancy.

In [ ]:
diff_actual = ts["residual_load"] - (ts["grid_load"] - ts["renewables"])
diff_fc = ts["fc_res"] - (ts["fc_grid_load"] - ts["fc_gen_wind_solar"])

report = (
    pd.DataFrame({"actual": diff_actual.abs(), "forecast": diff_fc.abs()})
    .describe(percentiles=[0.5, 0.95, 0.99])
    .T.round(4)
)
report["max_abs"] = [diff_actual.abs().max(), diff_fc.abs().max()]
report["n_above_0.5_MWh"] = [(diff_actual.abs() > 0.5).sum(), (diff_fc.abs() > 0.5).sum()]
display(report)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.hist(diff_actual, bins=41, color="C0")
ax.set_title(
    "residual_load  minus  (grid_load - wind_on - wind_off - solar)", fontsize=13, pad=10
)
ax.set_xlabel("difference (MWh)")
ax.set_ylabel("hours", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

The difference is bounded by **0.02 MWh** for the actuals and **0.01 MWh** for the forecasts, with
not a single hour above 0.5 MWh, and the histogram is a spike on a grid of two-decimal steps. On
a typical 50 000 MWh hour that residue is about 4 × 10⁻⁷ of the value — it is the CSV's
two-decimal format, nothing more.

So, stated plainly for the rest of the project to cite rather than re-open as an assumption:

> `residual_load` **is** `grid_load − (wind_on + wind_off + solar)`, exactly, and
> `fc_gen_wind_solar`'s actual counterpart **is** the summed wind and solar generation.

Two things follow. First, any near-perfect correlation between `grid_load`, the renewables and
`residual_load` in §9 is arithmetic, not a discovery, and must not be presented as a feature
finding. Second, modelling `residual_load` directly and modelling load-minus-renewables are the
same problem, so the choice between them is about error structure, not about definitions.

---

## 6 · Conventions and correctness

**This section is a contract, not a private choice.** The week convention, the bin label side,
the ISO grouping rule, the reporting units, the season definition and the descriptive-slice
policy fixed here are inherited by [`specs/02-Deep-EDA.md`](../specs/02-Deep-EDA.md) and by the
modeling work, which cite them rather than re-deriving them.

### 6.1 · The week convention: ISO, Monday start, Sunday end

In pandas, `"W"` is an alias for `"W-SUN"` — weeks *ending* Sunday, which is exactly ISO
Monday-start weeks. The alias reads as though it meant the opposite, so it is worth asserting
rather than trusting. But it has to be the *right* assertion: `to_period("W").start_time` is a
Monday for every bin **by construction**, including partial ones, so testing that would pass
without testing anything. The meaningful check is on the first *data* timestamp that falls into
each complete bin.

In [ ]:
weekly_bins = ts.index.to_period("W")
first_ts_in_bin = pd.Series(ts.index, index=weekly_bins).groupby(level=0).min()

complete_weeks = _complete_periods(ts.index, "W")
checked = first_ts_in_bin.loc[complete_weeks]

assert (checked.dt.dayofweek == 0).all() and (checked.dt.hour == 0).all(), \
    "a complete weekly bin does not open on Monday 00:00"
print(f"{len(checked):,} complete weekly bins, every one opening Monday 00:00")

# The counter-example that shows the test is not vacuous: the excluded first bin.
excluded_bin, excluded_ts = first_ts_in_bin.index[0], first_ts_in_bin.iloc[0]
print(
    f"excluded first bin {excluded_bin} opens {excluded_ts:%Y-%m-%d}, a {excluded_ts:%A} "
    "-- the record starts mid-week, which is why that bin is dropped rather than trusted"
)

244 complete weekly bins, every one opening Monday 00:00. The first bin is deliberately excluded
because the record opens on **Saturday 2022-01-01** — and printing that is what gives the
assertion teeth: a test that only ever saw Mondays would pass whether or not the convention held.

### 6.2 · Which edge of the week the label refers to

The two routes to a weekly aggregate label the same week with **different dates**:

In [ ]:
first_week = ts.index.to_period("W")[0]
print(f"the week bin {first_week}")
print(f"  period_mean    labels it {first_week.start_time:%Y-%m-%d} ({first_week.start_time:%A}) -- period START")
print(f"  resample('W')  labels it {first_week.end_time:%Y-%m-%d} ({first_week.end_time:%A}) -- RIGHT EDGE")

**Convention for this project: weekly x values are the Monday (the period start),** because every
weekly aggregate here goes through `period_mean`. Weekly axes are labelled accordingly, and §6.3
relabels `.resample()`'s output before comparing indexes for exactly this reason. An unlabelled
weekly axis is ambiguous by a six-day offset.

### 6.3 · Is `period_mean` correct?

`period_mean` is a plain calendar-period mean plus **one** extra rule. Before the rest of the
notebook leans on it, test it once against a plain `.resample()`.

These are the **only two `.resample()` calls in this notebook**. Everywhere else, weekly and
monthly aggregation goes through `period_mean` or `period_energy`. Note the `"ME"` spelling for
the monthly case: this project runs pandas 3, where `"M"` has been removed as an offset alias and
`.resample("M")` raises `ValueError`. `to_period("M")` is unaffected, so `period_mean(s, "M")`
itself still works — only the comparison side needs the new spelling.

In [ ]:
for freq, resample_alias in [("W", "W"), ("M", "ME")]:
    pm = period_mean(ts["grid_load"], freq)
    res = ts["grid_load"].resample(resample_alias).mean()

    # 1. Relabel before comparing anything: period_mean labels the start, resample the right edge.
    res_relabelled = res.copy()
    res_relabelled.index = res.index.to_period(freq).start_time

    # 2. On the periods both produce, the values must agree.
    shared = pm.index.intersection(res_relabelled.index)
    assert np.allclose(pm.loc[shared], res_relabelled.loc[shared])

    # 3. Exactly the INCOMPLETE periods are dropped -- expected set computed, not hard-coded.
    #    Restated from first principles: only the first and last period CAN be partial, so test
    #    each of those two against the data bounds.
    edges = dict.fromkeys([ts.index.min().to_period(freq), ts.index.max().to_period(freq)])
    expected_dropped = pd.PeriodIndex(
        [
            p for p in edges
            if p.start_time < ts.index.min() or p.end_time > ts.index.max() + pd.Timedelta("1h")
        ],
        freq=freq,
    )
    actual_dropped = res_relabelled.index.difference(pm.index)
    assert set(expected_dropped.start_time) == set(actual_dropped)

    print(f"{freq}: period_mean keeps {len(pm)}, resample produces {len(res)}, "
          f"agreeing on all {len(shared)} shared periods")

    # 4. Print each dropped edge next to a complete neighbour, so the artefact is visible in MW.
    for p in expected_dropped:
        neighbour = p + 1 if p.start_time < ts.index.min() else p - 1
        dropped_val = res_relabelled.loc[p.start_time]
        neighbour_val = res_relabelled.loc[neighbour.start_time]
        print(f"   dropped {p}: {dropped_val:>10,.0f} MW   vs neighbour {neighbour}: "
              f"{neighbour_val:>10,.0f} MW   ({dropped_val / neighbour_val - 1:+.1%})")
    print()

`period_mean` is a plain calendar-period mean plus one rule: **drop periods the data does not
fully cover**. The test above confirms both halves of that claim — the arithmetic is identical to
`.resample().mean()` on every period the two share, and the periods it drops are exactly the
incomplete ones, with the expected set computed from the data bounds rather than hard-coded.

**The rule bites asymmetrically here**, which is why "drop the first and last period" is the wrong
way to describe it. The record starts at exactly `2022-01-01 00:00`:

- **Weekly** — both edges go. 2022-01-01 is a Saturday, so the opening week is partial, and the
  record ends on a Wednesday, so the closing week is too. 244 of 246.
- **Monthly** — only the trailing edge goes. January 2022 is *complete*, because the data starts
  precisely on the month boundary. 56 of 57.

The printed values show the size of the artefact being avoided, and they correct a natural
assumption: the partial edges are **not** always dips. The opening week reads ~23 % *low* — it is
two public holidays and a weekend, nothing else. The closing week reads ~8 % *high*, because it
covers Monday to Wednesday only, three working days with no weekend to pull the mean down. The
distortion is signed by whichever days the partial period happens to contain, so "fake edge dip"
understates the problem: the edge can lie in either direction.

This test runs **once**, here, as a correctness check. It is not repeated for every later
aggregation.

### 6.4 · Does month length contaminate our comparisons?

`period_energy` (§2) exists so that month length cannot silently distort month-over-month
comparison. It returns two units — **MWh per day** (period sum ÷ calendar days) and **average
MW** (period sum ÷ hours *actually present*) — and drops incomplete periods on the same rule as
`period_mean`.

The table below puts four aggregation variants side by side for `grid_load`, with both
denominators exposed. It is shown with `drop_incomplete=False` so that the period the rule
discards is visible rather than merely described.

In [ ]:
pe = period_energy(ts["grid_load"], "M", drop_incomplete=False)
hourly_mean = ts["grid_load"].groupby(ts.index.to_period("M")).mean()

tab = pe.assign(
    hourly_mean=hourly_mean.to_numpy(),
    naive=pe["mwh_per_day"] / 24,  # the naive denominator: sum / (24 * days)
)
tab["naive_err_%"] = (100 * (tab["naive"] / tab["avg_mw"] - 1)).round(4)
tab["complete"] = tab.index.isin(period_energy(ts["grid_load"], "M").index)
tab = tab[["hourly_mean", "avg_mw", "mwh_per_day", "naive", "days", "hours", "naive_err_%", "complete"]]

# The negative result, stated as an assertion: not "close", identical.
assert (tab["hourly_mean"] - tab["avg_mw"]).abs().max() == 0.0

print("first three months:")
display(tab.head(3).round(2))

print("every month where hours != 24 x days -- the only rows where the naive denominator lies:")
display(tab[tab["hours"] != 24 * tab["days"]].round(2))

In [ ]:
sums = ts["grid_load"].groupby(ts.index.to_period("M")).sum()
jan, feb = pd.Period("2022-01", "M"), pd.Period("2022-02", "M")

print("February 2022 vs January 2022, same data, two aggregations:")
print(f"  raw monthly sum : {100 * (sums[feb] / sums[jan] - 1):+.1f} %   "
      f"({sums[jan]:,.0f} -> {sums[feb]:,.0f} MWh)")
print(f"  MWh per day     : "
      f"{100 * (pe['mwh_per_day'][feb.start_time] / pe['mwh_per_day'][jan.start_time] - 1):+.1f} %")
print(f"  average MW      : "
      f"{100 * (pe['avg_mw'][feb.start_time] / pe['avg_mw'][jan.start_time] - 1):+.1f} %")

Four conclusions, one of which is a negative result worth writing down:

1. **The hourly mean and average MW are the same number, always.** Not approximately — the
   assertion above is an exact `== 0.0` over all 57 months. Both divide by hours present, and
   since a value in MWh over a one-hour interval *is* the average MW, the two are the same
   quantity by construction.
2. **MWh per day is that number rescaled by days-per-month.** For *means*, month length cancels
   completely.
3. **The distortion the helper guards against appears only under the naive
   `sum / (24 × days)` denominator, and only where hours are missing.** The five March months
   hold 743 hours instead of 744 and read about **0.13 % low**. September 2026 holds 216 hours
   against 30 days and reads ~70 % low — but the edge rule drops it anyway, so it never reaches
   a plot.
4. **It is when we aggregate *sums* that ignoring month length distorts things**, and there the
   effect is large enough to invent a trend that is not there: February 2022 looks like an
   **8.6 % collapse** against January on raw sums, and is a **1.2 % rise** on MWh per day. The
   entire apparent drop is three fewer days.

So the mean-based views used throughout this notebook were fine all along. That is the honest
finding, and it is worth recording precisely so nobody re-opens the question later.

### 6.5 · Project conventions fixed here

**Reporting units.** **Average MW** for load and generation *levels*; **MWh per day** only where
the quantity is genuinely an energy volume. Every plot states which one it shows. This is why
`ylabel` is a required argument in the `style_timeseries` and `seasonal_plot` copies in §2 — the
originals default it to `"(MWh)"`, which would let a plot silently mislabel its own units.

**Seasons.** Meteorological seasons, with **December assigned to the following year's winter**:

In [ ]:
display(
    ts.groupby(["season_year", "season"], observed=True).size().unstack(fill_value=0)
)
print("season_year = year + (month == 12)")

winter = Dec/Jan/Feb, spring = Mar/Apr/May, summer = Jun/Jul/Aug, autumn = Sep/Oct/Nov, with
`season_year = year + (month == 12)`.

The December rule is not cosmetic. Under the alternative — December stays in its own calendar
year — winter 2022 would be January + February + December 2022, three months that never occurred
consecutively, and it would *look complete* when it is not. Under our rule, winter 2022 is
visibly Jan + Feb only — the table above shows **1 416 hours against ~2 160** for a full winter —
which is the honest representation of a record that starts on 1 January. Autumn 2026 is flagged
by the same mechanism at 216 hours. And because the data ends before December 2026, no orphan
2027 winter group arises.

**Descriptive slices.** Showing "the tail hours" requires selecting them, which looks like
thresholding but is not. The policy, stated here once so later sections do not collide with the
no-threshold rule:

> Rank-based slices — top/bottom 1 % by rank, largest-N by magnitude, longest run — are used
> **for description only**. No boolean column is created, nothing is persisted, and no slice is
> presented as a risk definition. Slices are computed inside the plotting cell and never added to
> `ts`.

The line is crisp: it becomes a threshold the moment a boolean column is created or a slice is
named a risk case. Defining the risk flag is the modeling spec's job, and §11's column assertion
is the mechanical check that this notebook did not quietly do it first.

---

## 7 · Univariate description

All eight series, described one at a time: summary statistics, the full-period level, and the
shape of the distribution. The three SMARD `fc_*` forecast columns are treated here as **ordinary
series** — how well they predict their actuals is a forecast-benchmark question, and that belongs
to [spec 02](../specs/02-Deep-EDA.md), not here.

### 7.1 · Summary statistics

In [ ]:
display(
    ts[SERIES]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T.drop(columns="count")
    .round(1)
)

Four things stand out.

**The renewables are hugely more variable than load.** `wind_on` has a standard deviation
(~9 700) almost as large as its mean (~12 400), and `solar`'s std exceeds its mean outright. By
contrast `grid_load` sits at ~53 300 ± 9 300 — it moves within a band, while generation swings
between nothing and tens of thousands of MW.

**`solar` is strongly zero-inflated by construction.** Its 25th percentile is ~6 MWh and its
median ~312 MWh, against a maximum of 58 056: half the hours in the year are night or near-night.
This is a property of the physical process, not a data defect (§5.5), but it makes the mean a
poor summary of solar and it is why the hour-of-day structure in §8 matters so much.

**Both residual load series reach below zero** — `residual_load` to −15 562 and `fc_res` to
−29 003. The percentiles locate the negative tail quite precisely: the **1st percentile is already
negative** (−3 501 and −3 487) while the 5th is solidly positive (+5 110 and +5 396), so somewhere
between 1 % and 5 % of hours sit below zero. §10 pins that down to 2.08 % and describes it
properly.

**The forecasts are not centred on their actuals.** `fc_grid_load` averages ~53 603 against
`grid_load`'s ~53 336, and `fc_res` averages ~30 628 against ~30 240. A few hundred MW of
systematic offset is visible even at this resolution — quantifying it is spec 02's job, but it is
worth noticing that the SMARD day-ahead forecasts are not unbiased.

### 7.2 · Level over the full period

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 17))

for ax, col in zip(axes.flat, SERIES):
    weekly = period_mean(ts[col], "W")
    ax.plot(weekly.index, weekly.to_numpy(), color="C0", linewidth=1.1)
    ax.axhline(0, color="0.6", linewidth=0.8, zorder=0)
    style_timeseries(ax, col, "average MW")
    ax.set_xlabel("week (labelled by its Monday)", color="grey", fontsize=9)

fig.suptitle(
    "All eight series, weekly means (incomplete edge weeks dropped)", fontsize=16, y=0.997
)
plt.tight_layout()
plt.show()

Weekly means, via `period_mean`, so the partial opening and closing weeks of §6.3 are not
plotted. Read at this resolution:

- **`grid_load`** has a clean annual cycle — winter high, summer low — riding on a visible
  **downward level shift after 2022**. §8.5 tests that properly rather than eyeballing it.
- **`solar`** is the most regular series in the set: a near-sinusoidal annual cycle whose summer
  peaks grow year on year, consistent with continued capacity build-out.
- **`wind_on` and `wind_off`** are the opposite — winter-weighted but dominated by weather, so
  the week-to-week scatter is large and no annual shape is as crisp as solar's.
- **`residual_load`** inherits load's annual cycle with the renewables subtracted out, which
  deepens its summer troughs over time. The horizontal zero line shows that weekly *means* never
  approach zero; the negative hours of §10 are an hourly phenomenon that weekly aggregation hides
  completely.
- The three **`fc_*`** series track their actuals closely enough to be visually indistinguishable
  at this scale, which is exactly why a proper error metric (spec 02) is needed to say anything
  useful about them.

### 7.3 · Distributions

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 13))

for ax, col in zip(axes.flat, SERIES):
    ax.hist(ts[col], bins=80, color="C0")
    ax.axvline(0, color="C3", linewidth=1.0)
    ax.set_title(col, fontsize=12, pad=8)
    ax.set_xlabel("MWh per hour")
    ax.set_ylabel("hours", color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

fig.suptitle("Hourly distribution of each series (red line = zero)", fontsize=16, y=0.999)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))

ax.boxplot(
    [ts[col] for col in SERIES],
    tick_labels=SERIES,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.25},
    medianprops={"color": "C1"},
)
ax.axhline(0, color="C3", linewidth=1.0)
ax.set_title("Spread and tails of each series", fontsize=15, pad=12)
ax.set_ylabel("MWh per hour", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="x", rotation=30)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.tight_layout()
plt.show()

The histograms and the boxplot say complementary things.

**Shapes split into three families.** `grid_load`, `fc_grid_load`, `residual_load` and `fc_res`
are broad and roughly unimodal — these are the series a model can reasonably target. `wind_on`,
`wind_off` and `fc_gen_wind_solar` are heavily right-skewed, piling up at low values with a long
thin tail of storm hours. `solar` is a different object again: a spike at the bottom of the range
holding more than half the record, then a wide shoulder. Averaging solar across all hours
describes no actual hour.

**The tails are asymmetric in a way that matters for this project.** The boxplot shows both
residual load series carrying whiskers and outliers in *both* directions — the high tail reaching
past 70 000 MWh and the negative tail crossing zero — whereas the generation series can only have
one. Those two directions are the two risk cases the project is built around (tight margins
above, renewable oversupply below), and §10 describes both.

**A boxplot is the wrong tool for `solar` and an informative one for `residual_load`.** Solar's
box is compressed against zero with thousands of "outliers" that are simply daylight hours — the
interquartile range describes the night, not the process. For `residual_load` the same plot is
genuinely diagnostic, because its distribution is close enough to symmetric that the whiskers
mean what they usually mean.

Note that no log scale appears anywhere above: `residual_load` and `fc_res` take negative values,
so log transforms are unusable for them, and applying one to the other series would have made the
panels mutually incomparable.

---

## 8 · Time structure

Four nested cycles and a trend: annual, weekly, daily, the calendar irregulars, and whatever
multi-year drift survives once seasonality is aggregated out. Every weekly and monthly aggregate
below goes through `period_mean` or `period_energy`, so the partial-edge artefact quantified in
§6.3 never reaches a plot.

### 8.1 · Annual seasonality

`seasonal_plot` is fed **pre-aggregated monthly means**, one row per (year, month). Handing it raw
hourly data would make seaborn bootstrap a confidence interval for every cell from ~700
observations — slow, and the band would describe within-month variation rather than the seasonal
shape we are after. Going through `period_mean` also drops the partial September 2026, so the
2026 line simply ends in August rather than plunging.

In [ ]:
for col in ["grid_load", "residual_load", "renewables", "solar"]:
    monthly = period_mean(ts[col], "M").to_frame(col)
    monthly["year"] = monthly.index.year
    monthly["month"] = monthly.index.month
    seasonal_plot(
        monthly,
        col,
        f"{col}: monthly mean by month of year, coloured by year (2026 ends in August)",
        "average MW",
    )

The years differ in **level and shape, but in different combinations per series**.

- **`grid_load`** keeps essentially the same shape every year — a winter-high, summer-low
  U — and the years are stacked apart in *level*: 2022 sits clearly above the rest. This is the
  level shift §8.5 quantifies.
- **`solar`** is the opposite: the shape is identical and strongly peaked in May–July, but the
  summer peak climbs year on year while the winter floor barely moves. That is capacity
  build-out expressed as a seasonal amplitude increase, not a level shift. (Capacity drift as a
  subject belongs to spec 02; noting the shape here is enough.)
- **`renewables`** inherits solar's growing summer bulge on top of wind's winter weighting,
  which flattens its annual profile over time.
- **`residual_load`** is where the two combine into the project's actual problem: it keeps
  load's U-shape, but the summer trough deepens year on year as solar grows, so the gap between
  2022 and 2026 is much wider in June–August than in December–January. Extrapolating that
  divergence is exactly what a residual load model has to get right.

### 8.2 · Weekly seasonality

In [ ]:
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
PROFILE_COLS = ["grid_load", "residual_load", "renewables"]

dow_profile = ts.groupby("dow")[PROFILE_COLS].mean()
dow_profile.index = DAY_NAMES

fig, ax = plt.subplots(figsize=(11, 5))
for col in PROFILE_COLS:
    ax.plot(dow_profile.index, dow_profile[col].to_numpy(), marker="o", label=col)
ax.set_title("Mean level by day of week", fontsize=15, pad=12)
ax.set_ylabel("average MW", color="grey")
ax.set_xlabel("day of week")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

weekend_contrast = ts.groupby("is_weekend")[PROFILE_COLS].mean().T
weekend_contrast.columns = ["weekday", "weekend"]
weekend_contrast["delta_%"] = (
    100 * (weekend_contrast["weekend"] / weekend_contrast["weekday"] - 1)
).round(1)
display(weekend_contrast.round(0))

The weekday/weekend split is the dominant weekly feature, and it is **a property of demand, not
of supply**. `grid_load` runs flat across Monday–Friday, drops on Saturday and bottoms out on
Sunday — weekends average **16 % below** weekdays. `renewables` is flat to within **1 %** across
all seven days, because weather does not know what day it is. The whole weekly cycle in
`residual_load` is therefore inherited from load, and is proportionally *larger* there
(**−27 %**): subtracting an unchanged renewable infeed from a smaller load amplifies the relative
swing.

For modelling this says: day-of-week matters, a binary weekend flag captures most of it, and it
should be applied to the load side of the problem rather than the generation side.

### 8.3 · Daily seasonality

The hour-of-day profile is shown for **`grid_load`**, split into four season panels with a
weekday and a weekend line in each. Keeping this on load is deliberate: the `residual_load`
hour-of-day profile is the *duck curve*, and its evolution is
[spec 02](../specs/02-Deep-EDA.md)'s subject. The two are complementary rather than duplicates.

In [ ]:
hour_profile = ts.groupby(["season", "is_weekend", "hour"], observed=True)["grid_load"].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True, sharex=True)

for ax, season in zip(axes.flat, SEASON_ORDER):
    for weekend, label, style in [(False, "weekday", "-"), (True, "weekend", "--")]:
        profile = hour_profile.loc[(season, weekend)]
        ax.plot(profile.index, profile.to_numpy(), style, marker="o", markersize=3, label=label)
    ax.set_title(season, fontsize=13, pad=8)
    ax.set_xlabel("hour of day (Europe/Berlin)")
    ax.set_ylabel("average MW", color="grey")
    ax.set_xticks(range(0, 24, 3))
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Mean grid load by hour of day, per season and weekday/weekend", fontsize=16)
plt.tight_layout()
plt.show()

Both expected features are visible, and they are seasonal in opposite ways.

**The winter evening peak** is unmistakable: winter weekdays rise to a sharp maximum around
18:00–19:00, when darkness and heating coincide with the tail of the working day. **The summer
midday dent** is the other signature — the summer weekday profile flattens and sags through
12:00–15:00 where winter peaks, because behind-the-meter solar is serving load that therefore
never appears as grid load. Spring sits between the two, autumn resembles winter with a smaller
amplitude.

The weekday/weekend gap is a near-constant vertical offset in every season with the **same
shape**: weekends have the same morning ramp and evening peak, simply lower and starting later.
That is a useful modelling fact — the weekend effect is close to multiplicative on the level, not
a different daily shape.

### 8.4 · Calendar effects: Christmas, New Year and public holidays

German **federal** holidays only — `holidays.country_holidays("DE")` with no `subdiv`. Grid load
is a national quantity, and state-specific holidays would make the comparison depend on which
Bundesland you happened to pick.

In [ ]:
years = range(ts.index.year.min(), ts.index.year.max() + 1)
de_holidays = holidays.country_holidays("DE", years=years)  # no subdiv -> federal only
holiday_dates = set(de_holidays)  # set(), not the dict: Series.isin on a dict subclass is not a contract

daily = ts.groupby("date")[["grid_load", "residual_load"]].mean()
daily.index = pd.DatetimeIndex(daily.index)
daily["dow"] = daily.index.dayofweek
daily["is_holiday"] = pd.Series(daily.index.date, index=daily.index).isin(holiday_dates)

print(f"{len(de_holidays)} federal holidays across {years.start}-{years.stop - 1}, "
      f"{daily['is_holiday'].sum()} of them inside the record")
print("distinct holiday names:", sorted({name for name in de_holidays.values()}))

In [ ]:
# Each holiday against non-holiday days of the SAME WEEKDAY within +/- 14 days.
rows = []
for day in daily.index[daily["is_holiday"]]:
    window = daily.loc[day - pd.Timedelta("14D"): day + pd.Timedelta("14D")]
    reference = window[(~window["is_holiday"]) & (window["dow"] == daily.at[day, "dow"])]
    if reference.empty:
        continue
    rows.append(
        {
            "holiday": de_holidays.get(day.date()),
            "date": day.date(),
            "weekday": day.day_name(),
            "load": daily.at[day, "grid_load"],
            "reference": reference["grid_load"].mean(),
            "delta_%": 100 * (daily.at[day, "grid_load"] / reference["grid_load"].mean() - 1),
        }
    )

holiday_effect = pd.DataFrame(rows)

by_name = (
    holiday_effect.groupby("holiday")
    .agg(n=("delta_%", "size"), mean_delta_pct=("delta_%", "mean"))
    .sort_values("mean_delta_pct")
    .round(1)
)
display(by_name)

Nine distinct federal holidays, 42 occurrences inside the record, and **every one of them
depresses grid load** against comparable same-weekday days in the surrounding four weeks. The
striking thing is how narrow the range is: from **−16.9 %** (Good Friday) to **−23.7 %**
(Christmas Day), with no holiday behaving qualitatively differently from the others. A holiday is
worth roughly a fifth of a day's load, whichever holiday it is.

That the list contains no Fronleichnam, Allerheiligen or Reformationstag is the check that the
no-`subdiv` call really did return federal holidays only; those three are state-level and would
have made the analysis depend on an arbitrary choice of Bundesland.

One caveat on method: for the Christmas holidays the ±14-day reference window is itself inside
the depressed Christmas period, so the measured delta **understates** the true effect. The
next cell measures that period directly instead.

In [ ]:
in_xmas = ((daily.index.month == 12) & (daily.index.day >= 24)) | (
    (daily.index.month == 1) & (daily.index.day <= 1)
)
# Baseline: ordinary working days of the same December, well clear of the holiday period.
baseline = (
    (daily.index.month == 12) & (daily.index.day <= 20) & (~daily["is_holiday"]) & (daily["dow"] < 5)
)

xmas = pd.DataFrame(
    {
        "xmas_period": daily.loc[in_xmas, "grid_load"].groupby(daily.index[in_xmas].year).mean(),
        "dec_baseline": daily.loc[baseline, "grid_load"].groupby(daily.index[baseline].year).mean(),
    }
)
xmas["delta_%"] = (100 * (xmas["xmas_period"] / xmas["dec_baseline"] - 1)).round(1)
display(xmas.round(0))

print("lowest-load days in the record:")
display(daily.nsmallest(8, "grid_load")[["grid_load", "dow", "is_holiday"]].round(0))

The Christmas/New Year period runs **18–25 % below the ordinary December working-day level** in
every year of the record — comparable in depth to a single public holiday but sustained over more
than a week, which makes it a far larger event in total. The effect is also shrinking: −24 % in
2022 against −18 % in 2025.

Two reading notes on the table. The rows group 1 January with its own calendar year, so each row
mixes late December of one year with New Year's Day of the next — the same December-boundary
problem the season rule in §6.5 exists to handle, left visible here rather than hidden. And the
2026 row has no baseline at all: the record ends on 09-09, so there is no December 2026, and that
row is New Year's Day 2026 standing alone.

**The lowest-load days are not the Christmas ones**, which is worth knowing before anyone builds
a feature on the assumption. They are **Sundays in spring and early summer** — Pentecost Sunday
2023, Easter Sunday 2024, Easter Sunday 2023 — where the weekly minimum coincides with the season
in which behind-the-meter solar suppresses grid load hardest. Christmas is a deeper *anomaly*
relative to its surroundings; a spring holiday Sunday is a lower *absolute* load.

For a model this says two things: a plain "is holiday" flag is not enough, because the Christmas
period is a multi-day regime in which the days *between* the holidays also behave like holidays;
and load minima are driven by the interaction of day-of-week with season, not by the holiday
calendar alone.

### 8.5 · Trend

Four complete years plus a stub will not carry a trend claim from annual means alone — 2026 stops
on 09-09, and September is a low-load month, so a naive annual mean for 2026 is biased downward by
the calendar rather than by anything physical. The comparison that works is **matching
1 January – 9 September windows**, year over year. That neutralises the partial final year instead
of merely labelling it.

In [ ]:
jan_sep = (ts["month"] < 9) | ((ts["month"] == 9) & (ts.index.day <= 9))
sliced = ts.loc[jan_sep]

trend = sliced.groupby("year")[["grid_load", "residual_load"]].mean()
trend["hours"] = sliced.groupby("year").size()
trend["load_vs_2022_%"] = (100 * (trend["grid_load"] / trend["grid_load"].iloc[0] - 1)).round(1)
display(trend.round(1))

full_year = ts.groupby("year")[["grid_load", "residual_load"]].mean()

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(trend))
ax.bar(x - 0.2, full_year["grid_load"].to_numpy(), width=0.4, label="whole calendar year")
ax.bar(x + 0.2, trend["grid_load"].to_numpy(), width=0.4, label="1 Jan - 9 Sep window")
ax.set_xticks(x)
ax.set_xticklabels([f"{y}\n(partial)" if y == trend.index.max() else str(y) for y in trend.index])
ax.set_title("Mean grid load per year: whole year vs. the matching Jan-Sep window", fontsize=15, pad=12)
ax.set_ylabel("average MW", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**The two series behave completely differently, and that is the main finding of this section.**

**`grid_load` shows a level shift, not a trend.** It drops **7.3 % from 2022 to 2023** and then
sits on a flat-to-slightly-rising plateau — −5.8 %, −6.3 %, −4.9 % against 2022 — through 2026. A
linear trend fitted across the whole record would describe that badly: it would under-predict 2022
and then over-predict a decline that stopped after one year. Whatever caused it (the 2022 energy
price shock being the obvious candidate) was a one-off step, and a model is better served by
letting recent history speak through lags than by extrapolating a slope.

**`residual_load` does have a trend, and it is the one this project is about.** In the same
matching windows it falls monotonically apart from one pause: 34 040 → 29 699 → 28 821 → 29 031 →
26 257 average MW, a **23 % decline from 2022 to 2026**. Only the first step of that is the load
shift; the rest is renewables displacing conventional generation year after year. So residual load
is falling faster and for longer than load itself, which is precisely why its negative tail grows
over the record (§10.2) and why a model trained on 2022–2023 behaviour would systematically
over-predict 2026.

The chart also shows why the window comparison was necessary. The whole-year bar for 2026 is not
comparable to the others (it stops in September, missing the high-load autumn months), and the
Jan–Sep bars are. Read only the second series across years.

Two mechanical notes: the 2024 window holds 24 more hours than the others because of 29 February,
so **means are compared, not sums**; and every window holds 6 047 rather than 6 048 hours because
of the spring DST gap from §5.2.

### 8.6 · Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

for col in ["grid_load", "residual_load"]:
    fig, axes = plt.subplots(2, 2, figsize=(15, 7.5))
    series = ts[col].to_numpy()

    plot_acf(series, lags=168, ax=axes[0, 0])
    plot_pacf(series, lags=168, method="ywm", ax=axes[0, 1])
    plot_acf(series, lags=48, ax=axes[1, 0])
    plot_pacf(series, lags=48, method="ywm", ax=axes[1, 1])

    # set_title AFTER the plot calls: statsmodels writes its own title into the axes.
    for ax, title in zip(
        axes.flat,
        [f"ACF - {col} - 168 lags", f"PACF - {col} - 168 lags (ywm)",
         "ACF - zoom 0-48", "PACF - zoom 0-48 (ywm)"],
    ):
        ax.set_title(title, fontsize=12, pad=8)
        ax.set_xlabel("lag (hours)")
        ax.set_ylabel("correlation")

    for ax in axes[0]:
        for lag in (24, 168):
            ax.axvline(lag, color="C3", linestyle=":", linewidth=1)

    plt.tight_layout()
    plt.show()

Both series show the same structure, and it is exactly the structure the daily and weekly sections
predicted.

**The 24 h peak dominates.** The ACF oscillates with a one-day period, staying high at every
multiple of 24 and dipping in between — this is the hour-of-day cycle of §8.3 seen from the
correlation side. **The 168 h peak is the second feature**: the ACF at one week is visibly higher
than at 144 h or 192 h, because the same hour of the same weekday recurs. The PACF concentrates
its mass in the first few lags plus isolated spikes at 24 and 168, which is the classic signature
saying *lag-1, lag-24 and lag-168 carry the information and the lags between them are mostly
redundant*.

`residual_load` decays faster than `grid_load` at long lags, because it inherits load's calendar
structure but adds weather noise from the renewables, which is much less persistent.

**One caveat on the lag axis.** The ACF treats consecutive rows as evenly spaced. Because of the
five missing DST hours (§5.2), every row after a spring switch is shifted by one hour relative to
true elapsed time, so lags spanning a switch are approximate. With five gaps in 41 107 rows the
distortion is negligible for reading peak positions, but it is the reason the axis is labelled
"lag (hours)" rather than presented as exact.

---

## 9 · Multivariate structure

How the series relate to each other, and which of those relations are candidate features. The
guiding caution for this whole section comes from §5.6: `residual_load` **is**
`grid_load − (wind_on + wind_off + solar)` exactly, so some of the strongest relationships here
are arithmetic and must not be dressed up as discoveries.

### 9.1 · Correlation matrix

In [ ]:
corr = ts[SERIES].corr()

fig, ax = plt.subplots(figsize=(9.5, 7.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Correlation between the eight series (hourly)", fontsize=15, pad=12)
plt.tight_layout()
plt.show()

Built from the `SERIES` constant rather than from every column of `ts`, so the derived calendar
columns cannot creep into a matrix that claims to describe measurements.

**Some of these numbers are not findings.** The entire `residual_load` row is constrained by the
identity verified in §5.6 — residual load is *defined* as load minus the renewables — so its
positive correlation with `grid_load` (+0.37) and its negative ones with wind (−0.45) and solar
(−0.51) are arithmetic consequences, not evidence that renewables drive residual load. Presenting
them as a discovery would be circular.

**Each actual tracks its own SMARD forecast at 0.96–0.97**, which sounds impressive and is mostly
a statement that both describe the same physical quantity. Whether that correlation corresponds to
a *useful* day-ahead error is a different question, and one this notebook deliberately does not
answer — see [spec 02](../specs/02-Deep-EDA.md).

**What is genuinely informative** is the top-left block:

- **Wind and solar are mildly *anti*-correlated** (−0.25), not independent. Meteorologically that
  is the expected sign — windy weather is cloudy weather — and it is good news: the two partially
  cover for each other, so their sum is smoother than either alone.
- **Demand and renewable supply are almost unrelated** at hourly resolution: `grid_load` against
  wind is +0.16 and against solar +0.18. The small positive sign for solar is the midday overlap
  of sunshine and working hours, nothing more. Supply and demand being this decoupled is precisely
  what makes residual load volatile — neither side compensates for the other.

### 9.2 · How residual load responds to renewable infeed

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

wind_total = ts["wind_on"] + ts["wind_off"]
panels = [
    ("wind + solar (renewables)", ts["renewables"]),
    ("wind only (on + offshore)", wind_total),
    ("solar only", ts["solar"]),
]

for ax, (label, x) in zip(axes, panels):
    hb = ax.hexbin(x, ts["residual_load"], gridsize=55, bins="log", cmap="Blues", mincnt=1)
    ax.axhline(0, color="C3", linewidth=1.0)
    ax.set_title(f"{label}\nr = {x.corr(ts['residual_load']):+.2f}", fontsize=12, pad=8)
    ax.set_xlabel(f"{label} (MWh per hour)")
    ax.grid(color="0.93", linewidth=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

axes[0].set_ylabel("residual load (MWh per hour)", color="grey")
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
fig.colorbar(hb, ax=axes, label="hours per bin (log scale)", fraction=0.025, pad=0.01)
fig.suptitle("Residual load against renewable infeed", fontsize=16)
plt.show()

All three relations are negative, as the identity requires, but they differ in a way that matters
for modelling.

**Against total renewables** (r = −0.78) the underlying relation is exactly linear with slope −1
— it must be, since residual load *is* load minus this quantity. So the correlation being −0.78
rather than −1.00 is itself the informative part: the vertical spread at any given infeed level is
not noise, it is the **entire variation of `grid_load`**, roughly 30 000 MW wide. Knowing
renewable infeed perfectly would still leave all of the demand uncertainty unresolved, and a
residual load model has to forecast both sides.

**Wind and solar contribute differently, and not in the order one might guess.** Solar has the
*stronger* hourly correlation (−0.51) despite being zero in half the record, while wind — which
runs day and night and reaches higher sustained levels — manages only −0.44. The reason is visible
in the panels: solar's cloud is bimodal, a dense column at zero covering every night hour plus a
separate daytime population, and within that daytime population the effect is very strong. Solar
does not push residual load down on average; it pushes it down **hard, in the middle of the day,
in summer** — which is exactly the mechanism behind the negative hours of §10.

The colour scale is logarithmic because the density is extremely uneven; on a linear scale the
night-time column would be the only visible feature of the solar panel.

### 9.3 · Lagged relations

In [ ]:
LAGS = [1, 24, 48, 168]

lag_corr = pd.DataFrame(
    {
        col: {f"lag {lag} h": ts[col].corr(ts[col].shift(lag)) for lag in LAGS}
        for col in ["residual_load", "grid_load"]
    }
).round(3)
display(lag_corr)

The empirical basis for candidate lag features, and the ordering is the point: **lag 168 h beats
lag 48 h in both series.** One week ago is the same weekday at the same position in the work
cycle; two days ago is neither. So a lag set of {1, 24, 168} is better motivated than a contiguous
block of recent lags.

The margin differs sharply between the two series, and that is itself informative. For
`grid_load` the weekly lag is dominant — **0.908 at 168 h against 0.598 at 48 h**, and it even
beats the 24 h lag (0.798). For `residual_load` the same ordering holds but only barely, 0.524
against 0.516, and both are far below `grid_load`'s values.

The reason is the decomposition of §9.1: `residual_load` carries load's calendar structure *plus*
weather noise from the renewables. Calendar structure repeats weekly; weather does not. So the
predictable part of residual load is weaker at every lag, and the weekly echo that dominates load
is nearly washed out. A model for residual load cannot lean on calendar lags as heavily as a load
model could.

**The DST caveat applies here too.** `.shift(n)` moves by *rows*, not by hours. For the five
windows straddling a spring switch (§5.2), `shift(24)` reaches back 23 wall-clock hours rather
than 24. With 5 affected positions out of 41 107 this does not move the correlations, but it is a
real defect that must be handled properly when these lags become model features rather than
descriptive statistics.

### 9.4 · Candidate features for the modeling spec

What this section recommends engineering — **as a recommendation, not as implemented code**. No
feature matrix is built here; that is explicitly out of scope.

**Calendar features.** Hour of day, day of week and a weekend flag are the highest-value items,
given that §8.2 and §8.3 show the daily and weekly cycles are the dominant structure. Month or
day-of-year captures the annual cycle. Hour of day should almost certainly be encoded cyclically
(sine/cosine) rather than as an integer, since hour 23 is adjacent to hour 0.

**Holiday features.** A federal-holiday flag, worth roughly −20 % on load (§8.4). But §8.4 also
showed a plain flag is insufficient: the Christmas–New Year window needs its own indicator, since
the ordinary days inside it behave like holidays without being any.

**Lags.** `residual_load` at 1 h, 24 h and 168 h, per §9.3 — deliberately *not* a contiguous block,
since lag 48 carries less than lag 168. Any implementation must handle the DST gaps by lagging on
the timestamp rather than on row position.

**Rolling aggregates.** Rolling means over 24 h and 168 h to capture the recent level, and a
rolling standard deviation as a volatility proxy. Time-based windows (`rolling("24h")`), not
row-count windows, for the same DST reason.

**Renewable aggregates.** The `renewables` sum, and wind and solar separately — §9.2 shows they
act through different mechanisms, so collapsing them to one number discards information. The
SMARD `fc_*` columns are themselves strong candidate features for a day-ahead model, since they
are genuinely available a day ahead; whether they are worth more than the baseline they represent
is spec 02's benchmark question.

**A caution carried forward.** §8.5 found that `residual_load` is falling 23 % across the record
while `grid_load` is flat. Any train/test split must respect time order, and a model fitted on
early data will over-predict late data unless the drift is handled explicitly.

---

## 10 · Residual load and its extremes

The target variable, described in detail — and its two tails, which are the risk cases the whole
project is built around: **high residual load** (imports, tight margins) and **negative residual
load** (renewable oversupply, negative prices, downward redispatch).

**This section describes. It does not define.** Per the policy fixed in §6.5, everything below
that selects "the extreme hours" does so by **rank** — largest-N, smallest-N, longest run — purely
to show where those hours sit on the calendar. No threshold is proposed, no cut-off is computed,
no boolean column is created, and nothing here is added to `ts`. Defining the risk flag is the
modeling spec's job, and §11 asserts mechanically that this notebook did not quietly do it first.

### 10.1 · The distribution

In [ ]:
res = ts["residual_load"]

shape = pd.Series(
    {
        "mean": res.mean(),
        "median": res.median(),
        "std": res.std(),
        "skew": res.skew(),
        "excess kurtosis": res.kurt(),
        "min": res.min(),
        "max": res.max(),
        "range": res.max() - res.min(),
        "min in std below median": (res.median() - res.min()) / res.std(),
        "max in std above median": (res.max() - res.median()) / res.std(),
    }
)
display(shape.round(2).to_frame("residual_load"))

fig, ax = plt.subplots(figsize=(13, 5))
ax.hist(res, bins=120, color="C0")
for q in [0.01, 0.99]:
    ax.axvline(res.quantile(q), color="0.35", linestyle="--", linewidth=1)
    ax.text(res.quantile(q), ax.get_ylim()[1] * 0.94, f" {q:.0%}", color="0.35", fontsize=9)
ax.axvline(0, color="C3", linewidth=1.6)
ax.text(0, ax.get_ylim()[1] * 0.8, "  zero", color="C3", fontsize=10)
ax.set_title("Distribution of hourly residual load", fontsize=15, pad=12)
ax.set_xlabel("residual load (MWh per hour)")
ax.set_ylabel("hours", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.tight_layout()
plt.show()

The distribution is **broad, single-peaked and close to symmetric** — a mild negative skew and
slightly light tails relative to a normal. That is a more forgiving shape than one might expect
for a risk variable: there is no heavy tail, no second mode, and the bulk of the mass sits in a
wide band either side of the median.

But symmetry of *shape* is not symmetry of *meaning*. The two tails reach comparable distances in
standard deviations, yet they are physically different events — the upper tail is a demand-driven
stress state, the lower one is a supply-driven oversupply state — and only the lower one has a
meaningful fixed landmark, **zero**, which the red line marks. That zero is not a statistical
feature of the distribution; it is where the sign of the problem flips.

Note again that no log scale is used and none can be: the series takes negative values.

### 10.2 · How much of the record is negative?

In [ ]:
negative = ts["residual_load"] < 0

print(f"negative hours overall : {negative.sum():,} of {len(ts):,}  ({negative.mean():.2%})")

by_year = pd.DataFrame(
    {
        "negative_hours": negative.groupby(ts["year"]).sum(),
        "hours": negative.groupby(ts["year"]).size(),
    }
)
by_year["share_%"] = (100 * by_year["negative_hours"] / by_year["hours"]).round(2)
by_year["note"] = ["", "", "", "", "partial year, to 09-09"]
display(by_year)

In [ ]:
neg_share = (100 * negative.groupby([ts["year"], ts["month"]]).mean()).unstack()

fig, ax = plt.subplots(figsize=(12, 3.6))
sns.heatmap(
    neg_share,
    annot=True,
    fmt=".1f",
    cmap="Reds",
    vmin=0,
    linewidths=0.5,
    cbar_kws={"label": "share of hours below zero (%)"},
    ax=ax,
)
ax.set_title("Share of hours with negative residual load, by year and month (%)", fontsize=14, pad=12)
ax.set_xlabel("month")
ax.set_ylabel("year")
plt.tight_layout()
plt.show()

**854 hours, 2.08 % of the record — and almost none of it is old.** The per-year progression is
the headline: **0.00 % in 2022**, then 0.58 %, 0.85 %, 2.31 %, and **8.70 % in 2026**. Negative
residual load went from non-existent to roughly one hour in twelve within four years, and 2026 is
only the partial January–September slice, which is the *solar-rich* part of the year.

A heatmap is used here rather than a line chart for a specific reason: the negative share is
structurally **exactly zero across all twelve months of 2022**, and a line chart renders that as a
flat baseline that dominates the axis and compresses everything interesting into the top strip.

**Blank cells are not zero.** October to December 2026 are blank because the record ends on
09-09 and those months contain no data at all. The 2022 row, by contrast, is a row of real,
measured `0.0` values. The distinction matters: one is "no negative hours occurred", the other is
"we do not know".

The seasonal pattern within the heatmap is as clear as the annual one. Negative hours concentrate
in **spring and summer** — the months with the most solar and the least demand — and are rare or
absent in November through January in every year. This is the seasonal signature of solar
oversupply, and it is the opposite season from the classic winter-peak stress case.

### 10.3 · When do the extremes happen?

Both tails, located on the calendar. The slices below are **rank-based** — the 1 % of hours with
the lowest values and the 1 % with the highest, taken by `nsmallest`/`nlargest` so that no
threshold value is ever computed or named. They exist only inside this cell.

In [ ]:
n_tail = round(0.01 * len(ts))
low_tail = ts["residual_load"].nsmallest(n_tail)   # descriptive slice, never persisted
high_tail = ts["residual_load"].nlargest(n_tail)

print(f"{n_tail:,} hours in each tail (1 % of the record by rank)")
print(f"  low  tail spans {low_tail.min():>10,.0f} .. {low_tail.max():>10,.0f} MWh")
print(f"  high tail spans {high_tail.min():>10,.0f} .. {high_tail.max():>10,.0f} MWh")

DIMENSIONS = [
    ("year", lambda idx: idx.year, None),
    ("month", lambda idx: idx.month, range(1, 13)),
    ("hour of day", lambda idx: idx.hour, range(24)),
    ("day of week", lambda idx: idx.dayofweek, range(7)),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

for ax, (label, extract, full_range) in zip(axes.flat, DIMENSIONS):
    frame = pd.DataFrame(
        {
            "lowest 1 % (oversupply)": pd.Series(extract(low_tail.index)).value_counts(normalize=True),
            "highest 1 % (tight margin)": pd.Series(extract(high_tail.index)).value_counts(normalize=True),
        }
    )
    if full_range is not None:
        frame = frame.reindex(list(full_range))
    frame = frame.sort_index() * 100

    x = np.arange(len(frame))
    ax.bar(x - 0.2, frame.iloc[:, 0].to_numpy(), width=0.4, label=frame.columns[0], color="C0")
    ax.bar(x + 0.2, frame.iloc[:, 1].to_numpy(), width=0.4, label=frame.columns[1], color="C3")
    ax.set_xticks(x)
    ax.set_xticklabels(
        DAY_NAMES if label == "day of week" else [str(v) for v in frame.index], fontsize=9
    )
    ax.set_title(f"by {label}", fontsize=12, pad=8)
    ax.set_ylabel("share of that tail (%)", color="grey")
    ax.set_xlabel(label)
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Where the two residual load tails sit on the calendar (1 % by rank each)", fontsize=16)
plt.tight_layout()
plt.show()

The two tails have **almost mirror-image calendar signatures**, which is the most useful thing in
this section.

**The low tail (oversupply)** is overwhelmingly recent — **76 % of it falls in 2026 alone** — and
concentrates in August, May, June and July, in the hours **11:00–15:00**, and at weekends:
Saturday and Sunday together hold **61 %** of it against a 28.6 % baseline. Every axis tells the
same story, the weekly demand minimum of §8.2 lining up with the daily solar maximum of §8.3.

**The high tail (tight margins) is the mirror image, and one number is categorical:
not a single hour of it falls on a weekend.** All 411 hours are Monday to Friday. It also spreads
much more evenly across years (2022 31 %, 2025 27 %, 2023 21 %) rather than trending, sits in
January, December, November and February, and peaks at **17:00–19:00** with a secondary cluster at
08:00 — the winter evening peak and morning ramp of §8.3, with little renewable infeed to offset
them.

That contrast is worth dwelling on. The low tail is **growing fast and seasonal**; the high tail
is **stable and structural**. One is a new phenomenon created by renewable build-out, the other is
the classic winter stress case that has always been there.

For the modeling spec this is a strong hint that **the two risk cases may not be one problem**.
They are driven by different mechanisms, occur in opposite seasons and at opposite hours, and a
single symmetric treatment of "extreme residual load" would blur two phenomena that a model could
learn separately. Whether to treat them as one target or two is a modeling decision — this
notebook only records that the evidence points toward two.

### 10.4 · Representative episodes

Selected by a **stated, reproducible, rank-based rule** rather than by eye:

1. **The longest run of consecutive negative `residual_load` hours.** Ties broken by earliest
   start — which matters here, because there are runners-up only one hour shorter.
2. **The highest 24-hour rolling mean.** The window is time-based (`rolling("24h")`), not
   row-based, so that the DST gaps of §5.2 cannot let a "24-row" window span 25 wall-clock hours.
   `min_periods=24` stops a partial window at the start of the record from winning.

Each is plotted with ±36 h of context and the concurrent wind, solar and grid load.

In [ ]:
# --- rule 1: longest consecutive negative run, ties broken by earliest start
blocks = (negative != negative.shift()).cumsum()
runs = (
    pd.DataFrame(
        {
            "length": negative.groupby(blocks).size(),
            "is_negative": negative.groupby(blocks).first(),
            "start": ts.index.to_series().groupby(blocks).min(),
        }
    )
    .query("is_negative")
    .sort_values(["length", "start"], ascending=[False, True])
)
print("longest negative runs (top 5), showing the tie the rule has to break:")
display(runs.head(5)[["start", "length"]].reset_index(drop=True))

run_start = runs.iloc[0]["start"]
run_end = run_start + pd.Timedelta(hours=int(runs.iloc[0]["length"]) - 1)

# --- rule 2: highest 24-hour rolling mean, on a TIME-based window
rolling_24h = ts["residual_load"].rolling("24h", min_periods=24).mean()
peak_end = rolling_24h.idxmax()
peak_start = peak_end - pd.Timedelta("23h")
print(f"\nhighest 24 h rolling mean: {rolling_24h.max():,.0f} MW over {peak_start} .. {peak_end}")

In [ ]:
EPISODE_COLS = ["residual_load", "grid_load", "wind_on", "wind_off", "solar"]
episodes = [
    (f"Longest negative run: {int(runs.iloc[0]['length'])} consecutive hours", run_start, run_end),
    (f"Highest 24 h rolling mean: {rolling_24h.max():,.0f} MW", peak_start, peak_end),
]

for title, start, end in episodes:
    window = ts.loc[start - pd.Timedelta("36h"): end + pd.Timedelta("36h"), EPISODE_COLS]

    fig, ax = plt.subplots(figsize=(15, 5.5))
    for col in EPISODE_COLS:
        ax.plot(
            window.index,
            window[col].to_numpy(),
            label=col,
            linewidth=2.2 if col == "residual_load" else 1.2,
            color="C3" if col == "residual_load" else None,
        )
    ax.axhline(0, color="0.35", linewidth=0.9)
    ax.axvspan(start, end, color="0.88", zorder=0, label="selected episode")

    style_timeseries(ax, f"{title}\n{start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} (+/- 36 h context)", "MWh per hour")
    # style_timeseries assumes a multi-year axis; a 4-day window needs day ticks.
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=(0, 6, 12, 18)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.set_xlabel("Europe/Berlin local time")
    ax.legend(frameon=False, ncol=6, fontsize=9)
    plt.tight_layout()
    plt.show()

The two episodes show the two mechanisms in their purest form.

**The longest negative run** — 12 hours on **Sunday 30 August 2026** — turns out to be a *handover*
between two renewable sources, not a pure solar event. It begins at 05:00 in darkness, pushed
below zero by onshore wind alone running near 31 000 MW. Solar then takes over as wind decays:
by 13:00 solar is at 34 500 MW while wind has fallen to 19 900, and residual load stays around
−13 000 throughout. The run ends at 16:00 when solar drops faster than demand recovers.

Three things had to coincide: **high wind, high solar, and a summer Sunday**, with grid load
sitting at 32 000–45 000 MW rather than a weekday's 55 000. That is why the deepest oversupply
events are weekend events — and it means a model cannot treat this as a solar phenomenon alone.

**The highest 24-hour mean** is a textbook **Dunkelflaute**: 10–11 January 2022, with grid load
averaging 65 300 MW (22 % above its record mean) while onshore wind averaged just **1 780 MW
against a record mean of 12 444 — about 14 % of normal** — offshore wind 1 324 against 2 902, and
solar 1 073 against 7 750. Both sides went wrong at once, and stayed wrong for a full day.

Nothing dramatic happens in any single hour of it. The severity is in the *sustained* combination,
which is exactly why a rolling-mean rule finds this episode and an hourly-maximum rule would
instead return an isolated evening peak.

The `axvspan` marks the selected window; the ±36 h of context on either side is what makes the
onset and recovery visible rather than just the episode itself.

### 10.5 · Tail behaviour, stated

Summarising what the last four sub-sections established, and **stopping there**:

- Residual load is broad, single-peaked and close to symmetric in shape, with a mild negative skew
  and no heavy tail. Its range spans roughly −15 600 to +71 000 MWh per hour.
- **The negative tail is real, growing fast, and seasonal.** It did not exist at all in 2022 and
  covered 8.7 % of hours in the partial 2026; 76 % of the lowest 1 % of hours fall in 2026 alone.
  It is a spring/summer, midday, **weekend** phenomenon — 61 % of that tail is Saturday or Sunday.
- **The high tail is stable, not growing, and seasonally opposite.** It sits in winter, at the
  17:00–19:00 evening peak, and **exclusively on weekdays — not one of its hours is a weekend
  hour.** It is driven by peak demand meeting little renewable infeed.
- The two tails are **different physical events with mirror-image calendar signatures**, not two
  ends of one symmetric risk. One is new and created by renewable build-out; the other is the
  classic winter stress case.
- The characteristic shape of each differs too. Negative episodes are short — the longest is
  12 hours — and need several factors to coincide at once (wind *and* solar *and* low weekend
  demand). High-residual episodes are sustained over a full day and are severe through duration
  rather than through any single hour.

**No threshold is proposed here, and none should be read into the above.** The rank-based slices
were descriptive devices for locating these hours on the calendar; the boundary between "extreme"
and "ordinary" is a modeling decision that depends on what the flag is for, and it is deferred
in full to the modeling spec.